## **Function Calling**

### **Loading genai**

In [38]:
import requests
from dotenv import load_dotenv
from google.genai import Client, types

In [39]:
load_dotenv()  # Load environment variables from .env file

client = Client()  # Initialize the Google GenAI client

- **Take Actions**: ```Interact with external systems using APIs, such as scheduling appointments, creating invoices, sending emails, or controlling smart home devices.```

- **Augment Knowledge**: ```Access information from external sources like databases, APIs, and knowledge bases.```

- **Extend Capabilities**: ```Use external tools to perform computations and extend the limitations of the model, such as using a calculator or creating charts.```

### **Augment knowledge**

In [3]:
personal_identifiers = {
    "type": "function",
    "name": "personal_identifiers",
    "description": "Extracts personal identifiers from the input text. (Name, age, hobby, education)",
    "parameters": {
        'type': 'object',
        'properties': {
            "name": {
                "type": "string", "description": "The name of the person. e.g., John Doe"
            },
            "age": {
                "type": "integer", "description": "The age of the person. e.g., 25"
            },
            "hobby": {
                "type": "string", "description": "The hobby of the person. e.g., reading, swimming, painting"
            },
            "education": {
                "type": "string", "description": "The education level of the person. e.g., bachelor's degree, master's degree"
            }
        },
        'required': ['name', 'age', 'hobby', 'education']
    }
}

In [4]:
interactions = client.interactions.create(
    model="gemini-3-flash-preview",
    input="Hello everyone, my name is John Doe. I am 25 years old and I love reading books. I have a bachelor's degree in Computer Science.",
    tools=[personal_identifiers]
)

In [5]:
for i in interactions:
    print(i)

('status', 'requires_action')
('model', 'gemini-3-flash-preview')
('agent', None)
('id', 'v1_ChctZU5XYXRHSkpQV21uc0VQc002V3lRbxIXLWVOV2F0R0pKUFdtbnNFUHNNNld5UW8')
('created', '2026-07-15T01:35:53Z')
('updated', '2026-07-15T01:35:53Z')
('system_instruction', None)
('tools', None)
('usage', Usage(cached_tokens_by_modality=None, grounding_tool_count=None, input_tokens_by_modality=[ModalityTokens(modality='text', tokens=205)], output_tokens_by_modality=None, tool_use_tokens_by_modality=None, total_cached_tokens=0, total_input_tokens=205, total_output_tokens=42, total_thought_tokens=183, total_tokens=430, total_tool_use_tokens=0))
('response_modalities', None)
('response_mime_type', None)
('previous_interaction_id', None)
('environment_id', None)
('service_tier', 'standard')
('webhook_config', None)
('steps', [FunctionCallStep(arguments={'name': 'John Doe', 'age': 25, 'hobby': 'reading books', 'education': "bachelor's degree in Computer Science"}, id='ujgboufi', name='personal_identifiers',

In [21]:
inputs = [
    "Hello everyone, my name is John Doe. I am 25 years old and I love reading books. I have a bachelor's degree in Computer Science.",
    "Hi, I'm Jane Smith. I'm 30 years old and my favorite hobby is painting. I hold a master's degree in Fine Arts.",
    "Greetings! My name is Michael Johnson. I'm 28 years old and I enjoy swimming. I have a bachelor's degree in Mechanical Engineering.",
    "Hey there, I'm Emily Davis. I'm 22 years old and I love hiking. I have a bachelor's degree in Environmental Science.",
    "Hello, I'm David Wilson. I'm 35 years old and my hobby is playing guitar. I hold a master's degree in Music."
]

In [28]:
results = []

for input_text in inputs:
    response = client.interactions.create(
        model="gemini-3-flash-preview",
        input=input_text,
        tools=[personal_identifiers]
    )

    fc_step = next(s for s in response.steps if s.type == "function_call")

    if fc_step.name == "personal_identifiers":
        result = fc_step.arguments
        results.append(result)

In [31]:
results

[{'age': 25,
  'education': "bachelor's degree in Computer Science",
  'hobby': 'reading books',
  'name': 'John Doe'},
 {'name': 'Jane Smith',
  'age': 30,
  'education': "master's degree in Fine Arts",
  'hobby': 'painting'},
 {'age': 28,
  'hobby': 'swimming',
  'name': 'Michael Johnson',
  'education': "bachelor's degree in Mechanical Engineering"},
 {'hobby': 'hiking',
  'age': 22,
  'education': "bachelor's degree in Environmental Science",
  'name': 'Emily Davis'},
 {'hobby': 'playing guitar',
  'name': 'David Wilson',
  'age': 35,
  'education': "master's degree in Music"}]

In [30]:
import pandas as pd

df = pd.DataFrame(results)
df

,age,education,hobby,name
0,25,bachelor's degree in Computer Science,reading books,John Doe
1,30,master's degree in Fine Arts,painting,Jane Smith
2,28,bachelor's degree in Mechanical Engineering,swimming,Michael Johnson
3,22,bachelor's degree in Environmental Science,hiking,Emily Davis
4,35,master's degree in Music,playing guitar,David Wilson


In [32]:
df.to_csv('personal_identifiers_results.csv', index=False)

### **Example 2**

In [6]:
def calculate_problem(a: float, b: float, sign: str) -> float:
    """
    Calculate the result of the problem based on the given inputs.

    Args:
        a (float): The first input value.
        b (float): The second input value.
        sign (str): The mathematical operator (+, -, *, /).

    Returns:
        float: The calculated result.
    """
    if sign == "+":
        return a + b
    elif sign == "-":
        return a - b
    elif sign == "*":
        return a * b
    elif sign == "/":
        return a / b
    else:
        raise ValueError("Invalid operator")

In [12]:
create_function_info = {
    'type': 'function',
    'name': 'calculate_problem',
    'description': 'Calculates the result of a mathematical problem based on the given inputs.',
    'parameters': {
        'type': 'object',
        'properties': {
            'a': {
                'type': 'number',
                'description': 'The first input value.'
            },
            'b': {
                'type': 'number',
                'description': 'The second input value.'
            },
            'sign': {
                'type': 'string',
                'description': 'The mathematical operator (+, -, *, /).'
            }
        },
        'required': ['a', 'b', 'sign']
    }
}

In [13]:
interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input="Calculate the result of 10 + 5.",
    tools=[create_function_info]
)

In [21]:
for step in interaction.steps:
    if step.type == "function_call" and step.name == "calculate_problem":
        result = step.arguments
        print(f"Result: {result}")

Result: {'a': 10, 'b': 5, 'sign': '+'}


In [ ]:
result = calculate_problem(*result.values())

In [24]:
print(result)

15


### Example 3

In [25]:
set_light_values_declaration = {
    "type": "function",
    "name": "set_light_values",
    "description": "Sets the brightness and color temperature of a light.",
    "parameters": {
        "type": "object",
        "properties": {
            "brightness": {
                "type": "integer",
                "description": "Light level from 0 to 100",
            },
            "color_temp": {
                "type": "string",
                "enum": ["daylight", "cool", "warm"],
                "description": "Color temperature",
            },
        },
        "required": ["brightness", "color_temp"],
    },
}

def set_light_values(brightness: int, color_temp: str) -> dict:
    """Set the brightness and color temperature of a room light."""
    return {"brightness": brightness, "colorTemperature": color_temp}

In [26]:
interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input="Turn the lights down to a romantic level",
    tools=[set_light_values_declaration],
)

fc_step = next(s for s in interaction.steps if s.type == "function_call")
print(fc_step)

arguments={'color_temp': 'warm', 'brightness': 30} id='mo1w555k' name='set_light_values' type='function_call' signature='EuwCCukCARFNMg/9wDqVKewOwB4aW6XUOGNM1kLbIuo8HZsZIWjKjXk/pOY2iXzbwOG6AIQ+lDhscAie0w1oOxfJegR5kecC5SxRD+ptKWRvQjgW8ybDk4rtBt4mhFgtPBlsEUw9fUeJgsyDGj69pNIW2vgZW6sO9h/kVAmAIXGKgjmBbsK1ShWhAqlvQ+gufutZv8S5kT/jHAPuCwFbDvyTvRqbhzaoAYSvbKLlKwvBTtWflIy8o3IfPMQXp9cvq9OJgPB/877Tj50ao9vJpZBUeQZ+ZTSv5wY8tJM1b+FPs5ypzxWtwPJaQLCJ04cvIr9beJv6Q32JTxvWV1W0gyNbRyN1kjrQ3oSofcdNQSi9DyZRLToYfrjxL+AIX65nnFv7Q47gezuUpSjRZtLTdFX2SryrlEM4TM+WYqzD82HANc83n6/ALa8vcGi/KfUT71WuLv8pDa9hjd6k1sVDAc4+gGsQEF9noVb9bVpqzw=='


In [27]:
fc_step = next(s for s in interaction.steps if s.type == "function_call")

if fc_step.name == "set_light_values":
    result = set_light_values(**fc_step.arguments)
    print(f"Function execution result: {result}")

Function execution result: {'brightness': 30, 'colorTemperature': 'warm'}


In [28]:
# Now, we will create a new interaction to send the function result back to the model for further processing or confirmation.
import json

final_interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input=[
        {
            "type": "function_result",
            "name": fc_step.name,
            "call_id": fc_step.id,
            "result": [{"type": "text", "text": json.dumps(result)}],
        }
    ],
    tools=[set_light_values_declaration],
    previous_interaction_id=interaction.id,
)

print(final_interaction.output_text)

OK. I've set the lights to a warm color temperature and dimmed them to 30% for a romantic atmosphere.


In [29]:
history = [
    {
        "type": "user_input",
        "content": [{"type": "text", "text": "Turn the lights down to a romantic level"}]
    }
]

interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    store=False,
    input=history,
    tools=[set_light_values_declaration],
)

for step in interaction.steps:
    history.append(step.model_dump())

fc_step = next(s for s in interaction.steps if s.type == "function_call")
if fc_step.name == "set_light_values":
    result = set_light_values(**fc_step.arguments)

history.append({
    "type": "function_result",
    "name": fc_step.name,
    "call_id": fc_step.id,
    "result": [{"type": "text", "text": json.dumps(result)}],
})

final_interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    store=False,
    input=history,
    tools=[set_light_values_declaration],
)

print(final_interaction.output_text)

OK. I've set the lights to a romantic level.


In [30]:
power_disco_ball = {"type": "function", "name": "power_disco_ball", "description": "Powers the disco ball.",
    "parameters": {"type": "object", "properties": {"power": {"type": "boolean"}}, "required": ["power"]}}
start_music = {"type": "function", "name": "start_music", "description": "Play music.",
    "parameters": {"type": "object", "properties": {"energetic": {"type": "boolean"}, "loud": {"type": "boolean"}}, "required": ["energetic", "loud"]}}
dim_lights = {"type": "function", "name": "dim_lights", "description": "Dim the lights.",
    "parameters": {"type": "object", "properties": {"brightness": {"type": "number"}}, "required": ["brightness"]}}

interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input="Turn this place into a party!",
    tools=[power_disco_ball, start_music, dim_lights],
    generation_config={"tool_choice": "any"},
)

for step in interaction.steps:
    if step.type == "function_call":
        args = ", ".join(f"{key}={val}" for key, val in step.arguments.items())
        print(f"{step.name}({args})")

power_disco_ball(power=True)
start_music(energetic=True, loud=True)
dim_lights(brightness=0.3)


In [31]:
get_weather_forecast_declaration = {
    "type": "function",
    "name": "get_weather_forecast",
    "description": "Gets the current weather temperature for a given location.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "The location"},
        },
        "required": ["location"],
    },
}

set_thermostat_temperature_declaration = {
    "type": "function",
    "name": "set_thermostat_temperature",
    "description": "Sets the thermostat to a desired temperature.",
    "parameters": {
        "type": "object",
        "properties": {
            "temperature": {
                "type": "integer",
                "description": "The temperature in Celsius",
            },
        },
        "required": ["temperature"],
    },
}

interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input="If it's warmer than 20°C in London, set the thermostat to 20°C, otherwise 18°C.",
    tools=[
        get_weather_forecast_declaration,
        set_thermostat_temperature_declaration,
    ],
)

for step in interaction.steps:
    if step.type == "function_call":
        print(f"Function to call: {step.name}")
        print(f"Arguments: {step.arguments}")
    elif hasattr(step, "content") and step.content:
         for part in step.content:
             if hasattr(part, "text"):
                 print(part.text)

Function to call: get_weather_forecast
Arguments: {'location': 'London'}


In [32]:
interaction.steps

[FunctionCallStep(arguments={'location': 'London'}, id='uhrhxovj', name='get_weather_forecast', type='function_call', signature='EpkECpYEARFNMg8XIacpZaIcS7awMMCJqroEGUzg3OJ7C1t6JYymWdJ3uvmFXY5Sb10+XdYl1tvFs+2OeZ+K/Y167mb6XxeZ057uBLbHNTx2MfmURxudIS7ltvIMbZB1PdjyT5KyQYSpnBTu1+hJhoIaRDQpJ36YlEbW4r5z3oeaiKImnb9kQvvSvKkZNE726BA4eQZ46leXSOe4+szESYluOZxE1QyLW1tzsxbpgV0UIkR7IkaNfteftOEHd+ue/8CbAy1WDAIqemU0CHWtE+/+xH3LppfMrJ1hGxLZ/KLC68uuon22QNDL+7EbRHt7ooxLIXXv4guaH88/W90PNo5Ba/h7VY5K0teayj4ZNTPVL2cCz1us6l7/CbxXyE65v/CSlIOO6PXQNO2Xn0+XgI4kpNjDG2hijlkUbNBY+Xzw/i1T8PXxCcqX1ivwvPztjKrTOtsEph+KWqIyzeQJdbGrx4S/kCxjo27KEO5xwmjGlzNbllJYnRduCZD9Q43cj6eh37YzbghmUqEHsNoxP8qGnRbA55VgbH412oXX0RVpDewIWMwZm4mpHOUUaL1dbH8aDdVOrlmmtoVIjjkItKCV9IrsAafB+phaj5aXdTLkQEjpM2lGI83Oc7d3fhPds61h5fRJVEC3aCKXXT2S9Lu+oYxE+dP4wQHq/c6DCVTkjrUxt4ae0C3L95iP+gN4NjaBIazV9tDcov/h')]

In [33]:
generation_config = {
    "tool_choice": {
        "allowed_tools": {
            "mode": "any",
            "tools": ["get_current_temperature"]
        }
    }
}

In [2]:
get_weather = {
    "type": "function",
    "name": "get_weather",
    "description": "Gets the weather for a requested city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city and state, e.g. Utqiaġvik, Alaska",
            },
        },
        "required": ["city"],
    },
}

tools = [
    {"type": "google_search"},
    get_weather               
]

interaction = client.interactions.create(
    model="gemini-3.1-flash-lite",
    input="What is the northernmost city in the United States? What's the weather like there today?",
    tools=tools
)

for step in interaction.steps:
    if step.type == "function_call":
        print(f"Function call: {step.name} (ID: {step.id})")
        result = {"response": "Very cold. 22 degrees Fahrenheit."}
        interaction_2 = client.interactions.create(
            model="gemini-3.1-flash-lite",
            previous_interaction_id=interaction.id,
            tools=tools,
            input=[{
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text", "text": json.dumps(result)}]
            }]
        )

        print(interaction_2.output_text)

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Your project has been denied access. Please contact support.', 'code': 'permission_denied'}}

## **Finally :)**